## This project implements a Transformer-based Neural Machine Translation model (english to spanish) built from scratch using PyTorch.
### Key components implemented in this project include:

Tokenization and vocabulary mapping

Positional encoding

Transformer encoder and decoder blocks

Attention masking (padding and causal masks)

Autoregressive decoding for translation

Training and evaluation pipeline

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("opus_books", "en-es")

In [ ]:
train_stream = dataset["train"]
dataset = dataset["train"].shuffle(seed=42).select(range(20000))

#data_subset = list(train_stream.take(10000))

In [ ]:
translation_pairs = [
    (ex["translation"]["en"], ex["translation"]["es"])
    for ex in dataset
]

target_language = "es"

In [ ]:
translation_pairs[0:5]

In [ ]:
import unicodedata, re
def clean_sentences(sentence):
    sentence=sentence.lower().strip()
    # preserve accents
    sentence=unicodedata.normalize("NFKC", sentence)
    # separate punctuation from words
    sentence=re.sub(r"([?.!,¿¡])",r" \1",sentence)
    sentence=re.sub(r"[^a-zA-Z0-9áéíóúñüÁÉÍÓÚÑÜ'?.!,¿¡]+", " ", sentence)
    sentence=re.sub(r"\s+", " ", sentence).strip()
    return sentence

In [ ]:
from collections import Counter
class MultiLanguageTokenizer():
    def __init__(self, max_vocab=5000):
        self.w2i={'<sos>':1, '<eos>':2, '<pad>':0, '<unk>':3}
        self.i2w={i:w for w,i in self.w2i.items()}
        self.word_count=Counter()
        self.vocab_size=4
        self.max_vocab=max_vocab
    def tokenize(self, sentences, min_freq=2):
        for sentence in sentences:
            tokens=sentence.split()
            self.word_count.update(tokens)
        most_common=self.word_count.most_common(self.max_vocab-len(self.w2i))
        idx=4
        for word, freq in most_common:
            if freq > min_freq :
                self.w2i[word]=idx
                self.i2w[idx]=word
                self.vocab_size+=1
                idx+=1
    def encode(self, sentence, max_len=300):
        tokens=sentence.split()
        encoded=[]
        encoded.append(self.w2i['<sos>'])
        for token in tokens:            
            encoded.append(self.w2i.get(token, self.w2i['<unk>']))     
        encoded.append(self.w2i['<eos>'])

        # while len(encoded)!=max_len:
        #     encoded.append(self.w2i['<pad>'])         
        
        return encoded
        
    def decoded(self, indices):
        return " ".join(self.i2w.get(idx,'<unk>') for idx in indices)
        
        

In [ ]:
def process_data(translation_pairs):
    src_language=[]
    tgt_language=[]
    for src, tgt in translation_pairs:
        src_language.append(src)
        tgt_language.append(tgt)

    # clean
    cleaned_src=[clean_sentences(src) for src in src_language]
    cleaned_tgt=[clean_sentences(tgt) for tgt in tgt_language]

    
    return cleaned_src, cleaned_tgt

In [ ]:
# test the functions
pairs=[('Kitty, on the contrary, was more active than usual and even more animated.',
  'Kitty, al contrario, estaba más activa a incluso más animada que nunca.')]

cleaned_src, cleaned_tgt=process_data(pairs)

In [ ]:
src_tokenizer=MultiLanguageTokenizer(5000)
src_tokenizer.tokenize(cleaned_src)

encoded_src=[src_tokenizer.encode(src) for src in cleaned_src]
print(encoded_src)


In [ ]:
tgt_tokenizer=MultiLanguageTokenizer(5000)
tgt_tokenizer.tokenize(cleaned_tgt)
encoded_tgt=[tgt_tokenizer.encode(tgt) for tgt in cleaned_tgt]
print(encoded_tgt)

In [ ]:
# prepare the full data
cleaned_src, cleaned_tgt=process_data(translation_pairs)
src_tokenizer.tokenize(cleaned_src)
tgt_tokenizer.tokenize(cleaned_tgt)
encoded_src=[src_tokenizer.encode(src) for src in cleaned_src]
encoded_tgt=[tgt_tokenizer.encode(tgt) for tgt in cleaned_tgt]

In [ ]:
print(encoded_src[:5])
print(encoded_tgt[:5])

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
class TranslationDataset(Dataset):
    def __init__(self, src, tgt):
        self.src=src
        self.tgt=tgt
        pass
    def __len__(self):
        return len(self.src)
    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]



# initialize the dataset
ds=TranslationDataset(encoded_src,encoded_tgt)

total_size=len(ds)
train_size=int(.8 * total_size)
val_size=total_size-train_size
tr_ds, val_ds=random_split(ds,[train_size, val_size])


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch, tgt_batch=zip(*batch)
    src_tensors=[torch.tensor(src) for src in src_batch]
    tgt_tensors=[torch.tensor(tgt) for tgt in src_batch]

    src_padded=pad_sequence(src_tensors, padding_value=0, batch_first=True)
    tgt_padded=pad_sequence(tgt_tensors, padding_value=0, batch_first=True)

    return src_padded, tgt_padded
    

# define src and target dataloaders
batch_size=32
tr_dl=DataLoader(tr_ds,batch_size=batch_size,shuffle=True, collate_fn=collate_fn)
val_dl=DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
# test the loaders
for src, tgt in tr_dl:
    print(src.shape)
    
    print(tgt.shape)
    break

## Create the Encoder-Decoder architecture

In [ ]:
import torch.nn as nn
# Create the padding mask
def set_padding_mask(seq,pad=0):
    return (seq==pad)
    
def set_causal_mask(size):
    mask=torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask
    
class PositionalEncoding(nn.Module):
    def __init__(self, max_seq_len, embedding_dimension):
        super().__init__()
        self.max_seq_len=max_seq_len
        self.embedding_dimension=embedding_dimension
        pem=torch.zeros(max_seq_len,embedding_dimension)
        positions=torch.arange(0,max_seq_len).unsqueeze(1).float()
        even_positions=torch.arange(0,embedding_dimension,2 ).float()
        exponential_term=torch.log(torch.tensor(10000.0))/embedding_dimension
        div_term=torch.exp(even_positions * -(exponential_term))
        # Sine to even indices
        pem[:,0::2]=torch.sin(positions * div_term)
        pem[:,1::2]=torch.cos(positions * div_term)
        self.register_buffer('pem', pem.unsqueeze(0))
    def forward(self, x):
        # shape of x is [batch_size, length_of_seq, embedding_dimension]
        length_of_seq=x.size(1)
        return x+self.pem[:,:length_of_seq, :]

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim,n_heads,ffn_hidden_dim, n_layers, max_len, dropout=.1):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,emb_dim,padding_idx=0)
        self.poe=PositionalEncoding(max_len,emb_dim)
        self.dropout=nn.Dropout(dropout)
        enc_layer=nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=n_heads,
            dim_feedforward=ffn_hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.trans_encoder=nn.TransformerEncoder(
            encoder_layer=enc_layer,
            num_layers=n_layers
        )
    # The encoder will see the source language only
    def forward(self, src):
        padding_mask=set_padding_mask(src)#(src==0)
        #print(f"Encoder padding  mask shape :{padding_mask.shape}")
        
        src=self.embedding(src)
        # positional embedding
        src=self.poe(src)
        src=self.dropout(src)
        #print(f'Encoder src shape: {src.shape}')
        context=self.trans_encoder(
            src, 
            src_key_padding_mask=padding_mask
        )
        return context, padding_mask
        

### Decoder will have token embedding-> posistional embedding -> self-attention -> cross-attention -> feed forward

In [ ]:
class Decoder(nn.Module):
    def __init__(self,  vocab_size, emb_dim,n_heads,ffn_hidden_dim, n_layers, max_len, dropout=.1):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,emb_dim, padding_idx=0)
        self.poe=PositionalEncoding(max_len, emb_dim)
        self.dropout=nn.Dropout(dropout)
        dec_layer=nn.TransformerDecoderLayer(
            d_model=emb_dim,
            nhead=n_heads,
            dim_feedforward=ffn_hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.trans_decoder=nn.TransformerDecoder(
            decoder_layer=dec_layer,
            num_layers=n_layers
        )
        self.ffn=nn.Linear(emb_dim, vocab_size)
        
    def forward(self, x, context, context_mask=None):
        padding_mask=set_padding_mask(x)
        # create causal mask
        causal_mask=set_causal_mask(x.size(1)).to(x.device)

        x=self.embedding(x)
        x=self.poe(x)
        x=self.dropout(x)
        decoded=self.trans_decoder(
            x,
            context,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask,
            memory_key_padding_mask=context_mask,
            memory_mask=None
            
        )
        outputs=self.ffn(decoded)
        return outputs

In [ ]:
class Translator(nn.Module):
    def __init__(self, vocab_size, emb_dim,n_heads,ffn_hidden_dim, n_layers, max_len, dropout=.1):
        super().__init__()
        self.encoder=Encoder(vocab_size, emb_dim,n_heads,ffn_hidden_dim, n_layers, max_len, dropout)
        self.decoder=Decoder(vocab_size, emb_dim,n_heads,ffn_hidden_dim, n_layers, max_len, dropout)
    def forward(self, src, tar):
        context, padding_mask=self.encoder(src)
        output=self.decoder(tar, context, padding_mask)
        return output
        

In [ ]:
src_tokenizer.vocab_size, tgt_tokenizer.vocab_size

In [ ]:
# initialize the model
import torch.optim as optim
vocab_size=src_tokenizer.vocab_size
emb_dim=512
n_heads=4
ffn_hidden_dim=2048
n_layers=4
max_len=600
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
translator=Translator(vocab_size,emb_dim,n_heads,ffn_hidden_dim,n_layers, max_len).to(device)

loss_fn=nn.CrossEntropyLoss(ignore_index=tgt_tokenizer.w2i['<pad>'])
optimizer=optim.Adam(translator.parameters(),lr=.001)

In [ ]:
def train_model(model,train_dl, test_dl,loss_fn,optimizer,epochs=10,device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=model.to(device)
    history={"train_acc":[], "test_acc":[],"train_loss":[],"test_loss":[]}
    
    for epoch in range(epochs):
        print(f"epoch: {epoch+1}")
        model.train()
        tr_loss=0
        total_train_size=0
        tr_correct=0
        for inputs, labels in train_dl:
            optimizer.zero_grad()
            if not torch.is_tensor(inputs):
                inputs = torch.tensor(inputs, dtype=torch.long)
            if not torch.is_tensor(labels):
                labels = torch.tensor(labels, dtype=torch.long)

            inputs=inputs.to(device)
            labels=labels.to(device)
            #print(inputs.shape)
            outputs=model(inputs, labels)
            #print(outputs.shape)
           
            if outputs.dim() == 3:
                B, T, V = outputs.shape                
                if labels.dim() == 2 and labels.shape[1] == T:
                    logits = outputs.reshape(-1, V)          
                    targets = labels.reshape(-1).long()    
                
                elif labels.dim() == 2 and labels.shape[1] == V:
                    outputs = outputs.permute(0, 2, 1)       
                    logits = outputs.reshape(-1, outputs.size(-1))
                    targets = labels.reshape(-1).long()
                else:
                    raise RuntimeError(f'unexpected shapes: outputs={outputs.shape}, labels={labels.shape}')
            elif outputs.dim() == 2:                
                logits = outputs
                targets = labels.reshape(-1).long()
            else:
                raise RuntimeError(f'unexpected outputs.dim()={outputs.dim()}')

            loss = loss_fn(logits, targets)
            
            loss.backward()
            optimizer.step()
    
            predicted=logits.argmax(dim=1)
            total_train_size+=targets.size(0)
            tr_correct+=(predicted==targets).sum().item()
            tr_loss+=loss.item()* targets.size(0)
    
            train_acc=100* tr_correct/total_train_size
            tr_avg_loss=tr_loss/total_train_size
            print(f"train acc: {train_acc} train loss: {tr_avg_loss}")
            
        model.eval()
        test_loss=0
        total_test_size=0
        test_correct=0
        with torch.no_grad():
            for inputs, labels in test_dl:
                if not torch.is_tensor(inputs):
                    inputs = torch.tensor(inputs, dtype=torch.long)
                if not torch.is_tensor(labels):
                    labels = torch.tensor(labels, dtype=torch.long)
                inputs=inputs.to(device)
                labels=labels.to(device)
    
                outputs=model(inputs, labels)
                if outputs.dim() == 3:
                    B, T, V = outputs.shape                
                    if labels.dim() == 2 and labels.shape[1] == T:
                        logits = outputs.reshape(-1, V)          
                        targets = labels.reshape(-1).long()    
                
                    elif labels.dim() == 2 and labels.shape[1] == V:
                        outputs = outputs.permute(0, 2, 1)       
                        logits = outputs.reshape(-1, outputs.size(-1))
                        targets = labels.reshape(-1).long()
                    else:
                        raise RuntimeError(f'unexpected shapes: outputs={outputs.shape}, labels={labels.shape}')
                elif outputs.dim() == 2:                
                    logits = outputs
                    targets = labels.reshape(-1).long()
                else:
                    raise RuntimeError(f'unexpected outputs.dim()={outputs.dim()}')
                loss=loss_fn(logits, targets)          

                #loss=loss_fn(outputs, labels)
                predicted=logits.argmax(dim=1)
                total_test_size+=targets.size(0)
                test_correct+=(predicted==targets).sum().item()
                test_loss+=loss.item()* targets.size(0)
        
                test_acc=100* test_correct/total_test_size
                test_avg_loss=test_loss/total_test_size
            print(f"test acc: {test_acc} test loss: {test_avg_loss}")
            
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['train_loss'].append(tr_avg_loss)
        history['test_loss'].append(test_avg_loss)
    return history
        

        

In [ ]:
history=train_model(translator,tr_dl, val_dl,loss_fn,optimizer,epochs=2,device=None)

In [ ]:
torch.save(translator.state_dict(),'/kaggle/working/translator.pth')

In [191]:
def translate_langauge(model, sentence, src_tokenizer,tgt_tokenizer, device, max_len, temparature=.5):
    model.eval()
    src_tokens=sentence.split()#src_tokenizer.tokenize([sentence])
    print(src_tokens)
    src_indices=[src_tokenizer.w2i.get(token, src_tokenizer.w2i['<unk>']) for token in src_tokens]
    # convert into tensor
    src_tensor=torch.tensor(src_indices).unsqueeze(0).to(device)
    print("tokens:", src_tokens)
    print("indices:", src_indices)
    print("src_tensor:", src_tensor)
    with torch.no_grad():
        memory, src_padding_mask=model.encoder(src_tensor)
        generated=[tgt_tokenizer.w2i['<sos>']]
        for _ in range(max_len):
            tgt_tensor=torch.tensor(generated).unsqueeze(0).to(device)
            output=model.decoder(tgt_tensor,
                                memory, src_padding_mask)
            logits=output[:,-1,:]
            logits[:, tgt_tokenizer.w2i['<pad>']] = -1e9
            logits[:, tgt_tokenizer.w2i[',']] = -1e9
            logits[:, tgt_tokenizer.w2i['<unk>']] = -1e9
            
            if temparature!=1.0:
                logits=logits/temparature
            probs=torch.softmax(logits,dim=1)
            if temparature > 1:
                nxt_token=torch.multinomial(probs,1).item()
            else:
                nxt_token = torch.multinomial(probs, 1).item()
            if nxt_token==tgt_tokenizer.w2i['<eos>']:
                break
            generated.append(nxt_token)
            
    #print(generated)
    words=[tgt_tokenizer.i2w[w] for w in generated if w not in [tgt_tokenizer.w2i['<eos>'], tgt_tokenizer.w2i['<sos>'], tgt_tokenizer.w2i['<pad>']]]
    print(words)
    return " ".join(words)   

In [194]:
test_sentences="I have finally done it."
cld_sentences=clean_sentences(test_sentences)
print(cld_sentences)

i have finally done it .


In [206]:
translate_langauge(translator,cld_sentences,src_tokenizer, tgt_tokenizer, device, 30,.7)

['i', 'have', 'finally', 'done', 'it', '.']
tokens: ['i', 'have', 'finally', 'done', 'it', '.']
indices: [11, 36, 1127, 229, 16, 6]
src_tensor: tensor([[  11,   36, 1127,  229,   16,    6]])
['.', 'un', '.', 'que', 'con', 'no', 'me', 'estaba', '.', 'de', 'el']


'. un . que con no me estaba . de el'

In [201]:
history

{'train_acc': [73.30461030021876, 73.58553864168618],
 'test_acc': [74.53554841014648, 74.53554841014648],
 'train_loss': [2.173048176677858, 2.125842756378567],
 'test_loss': [2.06223863456538, 2.0555280916006833]}